# 전처리 및 증강

In [ ]:
import os
from PIL import Image
from tqdm.auto import tqdm

# --- 경로 설정 ---
input_path = r"C:\Users\konyang\Desktop\2026_LAB\fundus_pp_dataset\1.merged_original"
output_path = r"C:\Users\konyang\Desktop\2026_LAB\data6\resize"

TARGET_SIZE = 224  # 각 눈 리사이즈 크기

os.makedirs(output_path, exist_ok=True)

valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
file_list = [f for f in os.listdir(input_path) if f.lower().endswith(valid_extensions)]

print(f"작업 시작: {len(file_list)} 파일")
print(f"처리 방식: 좌우안 분리 → 각각 {TARGET_SIZE}x{TARGET_SIZE} 리사이즈 → 합쳐서 저장 ({TARGET_SIZE*2}x{TARGET_SIZE})")

for filename in tqdm(file_list, desc="Processing"):
    try:
        img_path = os.path.join(input_path, filename)
        with Image.open(img_path) as img:
            img = img.convert('RGB')
            w, h = img.size

            # [Step 1: 가로 중앙 기준으로 좌안 / 우안 분리]
            mid = w // 2
            left_eye  = img.crop((0,   0, mid, h))
            right_eye = img.crop((mid, 0, w,   h))

            # [Step 2: 각각 224x224로 리사이즈]
            left_resized  = left_eye.resize((TARGET_SIZE, TARGET_SIZE), Image.LANCZOS)
            right_resized = right_eye.resize((TARGET_SIZE, TARGET_SIZE), Image.LANCZOS)

            # [Step 3: 좌우로 다시 합치기 → 448x224]
            merged = Image.new('RGB', (TARGET_SIZE * 2, TARGET_SIZE))
            merged.paste(left_resized,  (0,           0))
            merged.paste(right_resized, (TARGET_SIZE, 0))

            # [Step 4: 저장]
            merged.save(os.path.join(output_path, filename))

    except Exception as e:
        print(f"\n오류 발생 ({filename}): {e}")

print(f"\n완료. 저장 경로: {output_path}")
print(f"출력 이미지 크기: {TARGET_SIZE * 2} x {TARGET_SIZE} (좌안 {TARGET_SIZE}x{TARGET_SIZE} + 우안 {TARGET_SIZE}x{TARGET_SIZE})")

In [ ]:
import os
from PIL import Image
from tqdm.auto import tqdm

# --- 경로 및 설정 ---
input_path = r"C:\Users\konyang\Desktop\2026_LAB\data6\resize"
output_path = r"C:\Users\konyang\Desktop\2026_LAB\data6\aug"

# 회전 설정 (0도 ~ 350도, 10도 간격)
ROTATION_ANGLES = list(range(10, 360, 10))  # [10, 20, 30, ..., 350]

if not os.path.exists(output_path):
    os.makedirs(output_path)

valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
file_list = [f for f in os.listdir(input_path) if f.lower().endswith(valid_extensions)]

total_output = len(file_list) * len(ROTATION_ANGLES)
print(f"작업 시작: {len(file_list)} 파일 × {len(ROTATION_ANGLES)} 각도 = 총 {total_output}장 생성 예정")
print(f"적용 각도: {ROTATION_ANGLES}")

for filename in tqdm(file_list, desc="Processing Rotation Augmentation"):
    try:
        img_path = os.path.join(input_path, filename)
        with Image.open(img_path) as img:
            img = img.convert('RGB')
            w, h = img.size
            name, ext = os.path.splitext(filename)

            # [Step 1: 가로 중앙 기준으로 좌안 / 우안 분리]
            mid = w // 2
            left_eye  = img.crop((0,   0, mid, h))
            right_eye = img.crop((mid, 0, w,   h))

            for angle in ROTATION_ANGLES:
                # [Step 2: 좌안 / 우안 각각 동일 각도로 회전]
                # expand=False: 원본 캔버스 크기 유지, fillcolor='black': 빈 공간 검정 채움
                left_rotated  = left_eye.rotate(angle, resample=Image.BICUBIC, expand=False, fillcolor='black')
                right_rotated = right_eye.rotate(angle, resample=Image.BICUBIC, expand=False, fillcolor='black')

                # [Step 3: 회전된 좌안 + 우안 다시 합치기]
                merged = Image.new('RGB', (w, h))
                merged.paste(left_rotated,  (0,   0))
                merged.paste(right_rotated, (mid, 0))

                # [Step 4: 파일 저장] 예) filename_R010.jpg
                save_filename = f"{name}_A{angle:03d}{ext}"
                merged.save(os.path.join(output_path, save_filename))

    except Exception as e:
        print(f"\n오류 발생 ({filename}): {e}")

print(f"\n완료. 저장 경로: {output_path}")

In [ ]:
import numpy as np
import cv2
import os

# 경로 설정
input_path = r"C:\Users\konyang\Desktop\2026_LAB\data6\aug"
output_path = r"C:\Users\konyang\Desktop\2026_LAB\data6\aug_color"

# 출력 폴더가 없으면 생성
if not os.path.exists(output_path):
    os.makedirs(output_path)
    print(f"폴더 생성 완료: {output_path}")

# 증강 설정
ratios = np.arange(0.8, 1.3, 0.2)

image_files = [f for f in os.listdir(input_path)]

print(f"총 {len(image_files)}개의 이미지를 처리를 시작합니다.")

# 처리 루프
for i, filename in enumerate(image_files):
    # 이미지 로드
    img_path = os.path.join(input_path, filename)
    image = cv2.imread(img_path)

    if image is None:
        print(f"파일을 불러올 수 없습니다: {filename}")
        continue

    # 기본 이름과 확장자 분리
    name, ext = os.path.splitext(filename)

    # filp(image, 1) == 좌우, filp(image, 0) == 상하
    flipped = cv2.flip(image, 1)

    for ratio in ratios:
        # --- 1. 플립 + Contrast ---
        contrast_img = cv2.convertScaleAbs(flipped, alpha=ratio, beta=0)
        cv2.imwrite(os.path.join(output_path, f"{name}_Flip_Contrast_{ratio:.1f}{ext}"), contrast_img)

        # --- 2. 플립 + Brightness ---
        brightness_img = cv2.convertScaleAbs(flipped, alpha=1.0, beta=(ratio-1)*50)
        # (참고) ratio 0.5면 -25(어둡게), 1.0이면 0, 1.5면 +25(밝게)
        cv2.imwrite(os.path.join(output_path, f"{name}_Flip_Brightness_{ratio:.1f}{ext}"), brightness_img)

        # --- 3. 플립 + Color (채도 조절) ---
        hsv = cv2.cvtColor(flipped, cv2.COLOR_BGR2HSV).astype(np.float32)
        hsv[:, :, 1] = hsv[:, :, 1] * ratio # S(채도) 채널 조절
        hsv[:, :, 1] = np.clip(hsv[:, :, 1], 0, 255) # 범위를 0~255로 제한
        color_img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
        cv2.imwrite(os.path.join(output_path, f"{name}_Flip_Color_{ratio:.1f}{ext}"), color_img)

print("모든 증강 작업이 완료되었습니다!")

In [ ]:
import os
import cv2

# 원본 이미지 폴더
input_folder = r"C:\Users\konyang\Desktop\2026_LAB\data6\aug_color"
folder = r"C:\Users\konyang\Desktop\2026_LAB\data6\fundus_pp"

# 저장 폴더들
#out_r_gray   = os.path.join(folder, "3.rch_gray")
#out_g_gray   = os.path.join(folder, "4.gch_gray")
out_r_clahe  = os.path.join(folder, "5.rch_clahe")
out_g_clahe  = os.path.join(folder, "6.gch_clahe")

# 폴더 생성
for folder in [out_r_clahe, out_g_clahe]:
    os.makedirs(folder, exist_ok=True)

# 처리할 확장자
extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

# CLAHE 객체 생성
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

for file in os.listdir(input_folder):
    if not file.lower().endswith(extensions):
        continue

    img_path = os.path.join(input_folder, file)
    img = cv2.imread(img_path)

    if img is None:
        print(f"읽기 실패: {file}")
        continue

    # OpenCV는 BGR 순서
    b, g, r = cv2.split(img)

    # 1) R 채널 grayscale 저장
    #cv2.imwrite(os.path.join(out_r_gray, file), r)

    # 2) G 채널 grayscale 저장
    #cv2.imwrite(os.path.join(out_g_gray, file), g)

    # 3) R 채널 + CLAHE 저장
    r_clahe = clahe.apply(r)
    cv2.imwrite(os.path.join(out_r_clahe, file), r_clahe)

    # 4) G 채널 + CLAHE 저장
    g_clahe = clahe.apply(g)
    cv2.imwrite(os.path.join(out_g_clahe, file), g_clahe)

print("완료")

In [ ]:
import os

base_path = r"C:\Users\konyang\Desktop\2026_LAB\data6\fundus_pp"

folder_suffix = {
    "5.rch_clahe": "_rc",
    "6.gch_clahe": "_gc",
}

for folder_name, suffix in folder_suffix.items():
    folder_path = os.path.join(base_path, folder_name)

    if not os.path.exists(folder_path):
        print(f"[경고] 폴더를 찾을 수 없음: {folder_path}")
        continue

    file_list = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
    print(f"\n[{folder_name}] {len(file_list)}개 파일 처리 시작")

    renamed, skipped = 0, 0
    for filename in file_list:
        name, ext = os.path.splitext(filename)

        # 이미 suffix가 붙어있으면 스킵
        if name.endswith(suffix):
            skipped += 1
            continue

        new_filename = f"{name}{suffix}.jpg"
        src = os.path.join(folder_path, filename)
        dst = os.path.join(folder_path, new_filename)

        os.rename(src, dst)
        renamed += 1

    print(f"  완료: {renamed}개 변경, {skipped}개 스킵(이미 처리됨)")

print("\n전체 완료.")

In [ ]:
import os
import pandas as pd

aug_color_path = r"C:\Users\konyang\Desktop\2026_LAB\data6\aug_color"
files = [f for f in os.listdir(aug_color_path)
         if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif'))]

data = []
for f in files:
    name = os.path.splitext(f)[0]
    if name[0].isdigit():
        target = 'Normal'
    elif name[0].upper() == 'A':
        target = 'NAION'
    elif name[0].upper() == 'B':
        target = 'ON'
    else:
        print(f"⚠️ 알 수 없는 파일명: {name}")
        continue
    data.append({'filename': name, 'target': target})

df = pd.DataFrame(data)
save_path = r"C:\Users\konyang\Desktop\2026_LAB\data6\dataset\1.csv"
df.to_csv(save_path, index=False, encoding='utf-8-sig')

print(f"✅ 완료! 총 {len(df)}개")
print(df['target'].value_counts())

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

base_paths = [
    r"C:\Users\konyang\Desktop\2026_LAB\data6\fundus_pp\5.rch_clahe",
    r"C:\Users\konyang\Desktop\2026_LAB\data6\fundus_pp\6.gch_clahe"
]
npy_save_path   = r"C:\Users\konyang\Desktop\2026_LAB\data6\dataset\X"
output_csv_path = r"C:\Users\konyang\Desktop\2026_LAB\data6\dataset\Y.csv"
ref_csv_path    = r"C:\Users\konyang\Desktop\2026_LAB\data6\dataset\1.csv"
img_size = 224

os.makedirs(npy_save_path, exist_ok=True)
os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)

try:
    ref_df = pd.read_csv(ref_csv_path, encoding='utf-8-sig')
except Exception:
    ref_df = pd.read_csv(ref_csv_path, encoding='cp949')

ref_df.columns = [col.strip().replace('\ufeff', '') for col in ref_df.columns]

if 'filename' in ref_df.columns:
    ref_df['pure_name'] = ref_df['filename'].astype(str).str.strip()
    print(f"✅ CSV 로드 완료. 첫 번째 샘플 ID: {ref_df['pure_name'].iloc[0]}")
else:
    print(f"현재 컬럼명: {ref_df.columns.tolist()}")
    raise KeyError("CSV에 'filename' 컬럼이 없습니다.")

label_map_onehot = {
    'Normal': [1, 0, 0],
    'NAION':  [0, 1, 0],
    'ON':     [0, 0, 1]
}

final_data_list = []

for b_path in base_paths:
    if not os.path.exists(b_path):
        print(f"⚠️ 경로 없음: {b_path}")
        continue

    folder_name = os.path.basename(os.path.abspath(b_path))
    print(f"\n🚀 처리 중: {folder_name}")

    for root, dirs, files in os.walk(b_path):
        for img_name in tqdm(files, desc=f"   └ {folder_name}"):
            if not img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif')):
                continue

            # _rc / _gc 접미사 제거 후 CSV와 매칭
            stem = os.path.splitext(img_name)[0].strip()
            for suffix in ('_rc', '_gc'):
                if stem.endswith(suffix):
                    stem = stem[:-len(suffix)]
                    break
            img_base_id = stem

            match = ref_df[ref_df['pure_name'] == img_base_id]
            if not match.empty:
                try:
                    target_str = str(match['target'].values[0]).strip()
                    labels     = label_map_onehot.get(target_str)
                    if labels is None:
                        continue

                    full_img_path = os.path.join(root, img_name)
                    img = cv2.imread(full_img_path)
                    if img is None:
                        continue

                    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                    img = cv2.resize(img, (img_size, img_size))

                    save_name = f"{folder_name}_{img_name}.npy"
                    np.save(os.path.join(npy_save_path, save_name), img)

                    final_data_list.append({
                        'npy_file':      save_name,
                        'origin_folder': folder_name,
                        'target_name':   target_str,
                        'label1': labels[0],
                        'label2': labels[1],
                        'label3': labels[2]
                    })
                except Exception:
                    continue

if final_data_list:
    final_df = pd.DataFrame(final_data_list)
    final_df.to_csv(output_csv_path, index=False, encoding='utf-8-sig')
    print("-" * 30)
    print(f"✅ 작업 완료! 총 {len(final_df)}개 데이터 생성")
    print("📊 클래스별 분포:")
    print(final_df['target_name'].value_counts())
else:
    print("❌ 처리된 데이터가 없습니다.")

# 모델

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torchvision.ops import StochasticDepth
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from tqdm import tqdm

In [ ]:
device        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

num_classes   = 3
num_epochs    = 20
learning_rate = 1e-4
n_splits = 3
random_state=42
batch_size = 64
#num_workers = 0

CHANNEL_MODE = 'both'   # 'r', 'g', 'both' 중 선택

x_dir       = r"C:\Users\konyang\Desktop\2026_LAB\data6/dataset/X"
y_path       = r"C:\Users\konyang\Desktop\2026_LAB\data6/dataset/Y.csv"

# 라벨 매핑 — Y.csv one-hot 정의와 동일한 순서
# normal [1,0,0] → 정수 0  |  NAION [0,1,0] → 정수 1  |  ON [0,0,1] → 정수 2
label_map = {'normal': 0, 'NAION': 1, 'ON': 2}

print(f'device: {device}')

In [ ]:
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduced_dim):
        super().__init__()
        self.avgpool          = nn.AdaptiveAvgPool2d(1)
        self.fc1              = nn.Conv2d(in_channels, reduced_dim, kernel_size=1)
        self.activation       = nn.SiLU()
        self.fc2              = nn.Conv2d(reduced_dim, in_channels, kernel_size=1)
        self.scale_activation = nn.Sigmoid()

    def forward(self, x):
        scale = self.avgpool(x)
        scale = self.fc1(scale)
        scale = self.activation(scale)
        scale = self.fc2(scale)
        scale = self.scale_activation(scale)
        return x * scale

In [ ]:
class MBConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size,
                 stride, expand_ratio, se_ratio=0.25):
        super().__init__()
        self.use_residual = (in_channels == out_channels) and (stride == 1)
        expanded_ch  = in_channels * expand_ratio
        reduced_dim  = max(1, int(in_channels * se_ratio))

        block_layers = []
        if expand_ratio != 1:
            block_layers.append(nn.Sequential(          # block.0  Expand
                nn.Conv2d(in_channels, expanded_ch, 1, bias=False),
                nn.BatchNorm2d(expanded_ch),
                nn.SiLU()
            ))
            block_layers.append(nn.Sequential(          # block.1  Depthwise
                nn.Conv2d(expanded_ch, expanded_ch, kernel_size,
                          stride=stride, padding=kernel_size // 2,
                          groups=expanded_ch, bias=False),
                nn.BatchNorm2d(expanded_ch),
                nn.SiLU()
            ))
            block_layers.append(SEBlock(expanded_ch, reduced_dim))  # block.2  SE
            block_layers.append(nn.Sequential(          # block.3  Project
                nn.Conv2d(expanded_ch, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels)
            ))
        else:
            block_layers.append(nn.Sequential(          # block.0  Depthwise
                nn.Conv2d(expanded_ch, expanded_ch, kernel_size,
                          stride=stride, padding=kernel_size // 2,
                          groups=expanded_ch, bias=False),
                nn.BatchNorm2d(expanded_ch),
                nn.SiLU()
            ))
            block_layers.append(SEBlock(expanded_ch, reduced_dim))  # block.1  SE
            block_layers.append(nn.Sequential(          # block.2  Project
                nn.Conv2d(expanded_ch, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels)
            ))

        self.block            = nn.Sequential(*block_layers)
        # 가중치 없는 모듈이지만 torchvision 과 키 구조를 맞추기 위해 등록
        self.stochastic_depth = StochasticDepth(p=0.2, mode='row')

    def forward(self, x):
        out = self.block(x)
        if self.use_residual:
            out = self.stochastic_depth(out)
            out = out + x
        return out

In [ ]:
class EfficientNetB0(nn.Module):
    # (expand_ratio, out_channels, repeats, stride, kernel_size)
    MB_CONFIG = [
        (1,  16, 1, 1, 3),   # features.1
        (6,  24, 2, 2, 3),   # features.2
        (6,  40, 2, 2, 5),   # features.3
        (6,  80, 3, 2, 3),   # features.4
        (6, 112, 3, 1, 5),   # features.5
        (6, 192, 4, 2, 5),   # features.6
        (6, 320, 1, 1, 3),   # features.7
    ]

    def __init__(self, num_classes=1000):
        super().__init__()

        stages = [
            nn.Sequential(                          # features.0  Stem
                nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
                nn.BatchNorm2d(32),
                nn.SiLU()
            )
        ]

        in_ch = 32
        for expand_ratio, out_ch, repeats, stride, kernel_size in self.MB_CONFIG:
            stage = []
            for i in range(repeats):
                stage.append(MBConv(
                    in_ch, out_ch, kernel_size,
                    stride=stride if i == 0 else 1,
                    expand_ratio=expand_ratio
                ))
                in_ch = out_ch
            stages.append(nn.Sequential(*stage))   # features.1 ~ features.7

        stages.append(nn.Sequential(               # features.8  Head Conv
            nn.Conv2d(320, 1280, 1, bias=False),
            nn.BatchNorm2d(1280),
            nn.SiLU()
        ))

        self.features   = nn.Sequential(*stages)   # features.0 ~ features.8
        self.avgpool    = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),           # classifier.0
            nn.Linear(1280, num_classes) # classifier.1
        )

    def forward(self, x):
        x = self.features(x)    # (B, 1280, 7, 7)
        x = self.avgpool(x)     # (B, 1280, 1, 1)
        x = x.flatten(1)        # (B, 1280)
        x = self.classifier(x)  # (B, num_classes)
        return x

In [ ]:
def transplant_weights(custom_model):
    official_sd = models.efficientnet_b0(
        weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
    ).state_dict()
    custom_sd = custom_model.state_dict()

    matched, skipped, new_dict = [], [], {}
    for key in custom_sd:
        if key in official_sd and custom_sd[key].shape == official_sd[key].shape:
            new_dict[key] = official_sd[key]
            matched.append(key)
        else:
            new_dict[key] = custom_sd[key]
            reason = 'SHAPE MISMATCH' if key in official_sd else 'NOT FOUND'
            skipped.append(f'{reason} | {key}')

    custom_model.load_state_dict(new_dict, strict=True)

    return custom_model

In [ ]:
transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225])
])

train_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

val_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

In [ ]:
class FundusDataset(Dataset):
    def __init__(self, df, npy_path, transform=None):
        self.df        = df.reset_index(drop=True)
        self.npy_path  = npy_path
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = np.load(os.path.join(self.npy_path, row['npy_file']))
        label = int(row['temp_label'])
        if self.transform:
            img = self.transform(img)
        return img, label


# ← 클래스 밖으로 꺼냄
def filter_by_channel(df, mode):
    if mode == 'r':
        return df[df['npy_file'].str.endswith('_rc.jpg.npy')].reset_index(drop=True)
    elif mode == 'g':
        return df[df['npy_file'].str.endswith('_gc.jpg.npy')].reset_index(drop=True)
    else:
        return df.reset_index(drop=True)


train_val_df_exp = filter_by_channel(train_val_df, CHANNEL_MODE)
test_df_exp      = filter_by_channel(test_df,      CHANNEL_MODE)

print(f"[{CHANNEL_MODE}채널] train_val: {len(train_val_df_exp)}개 / test: {len(test_df_exp)}개")

In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
import numpy as np

random_state = 42
label_map = {'Normal': 0, 'NAION': 1, 'ON': 2}

full_df = pd.read_csv(y_path)
full_df['temp_label'] = full_df['target_name'].map(label_map)

def extract_patient_id(npy_filename):
    # 예: "5.rch_clahe_1.jpg_merged_A010_Flip_Brightness_0.8_rc.jpg.npy"
    # folder prefix 제거
    for prefix in ('5.rch_clahe_', '6.gch_clahe_'):
        if npy_filename.startswith(prefix):
            npy_filename = npy_filename[len(prefix):]
            break
    # → "1.jpg_merged_A010_Flip_Brightness_0.8_rc.jpg.npy"
    # 맨 앞 숫자(환자 ID) 추출
    return npy_filename.split('.')[0]  # → "1"

full_df['patient_id'] = full_df['npy_file'].apply(extract_patient_id)

# ── 클래스별 환자 현황 확인
print("=== 클래스별 고유 환자 수 ===")
class_patient_counts = (
    full_df.groupby('target_name')['patient_id']
    .nunique()
    .reset_index()
    .rename(columns={'patient_id': 'unique_patients'})
)
print(class_patient_counts)

# ── 클래스별로 test 환자를 강제 확보 (10% 비율)
np.random.seed(random_state)

test_patient_ids = set()
TEST_RATIO = 0.1

for label_name in full_df['target_name'].unique():
    class_df = full_df[full_df['target_name'] == label_name]
    unique_patients = class_df['patient_id'].unique()
    n_test = max(1, int(len(unique_patients) * TEST_RATIO))
    selected = np.random.choice(unique_patients, size=n_test, replace=False)
    test_patient_ids.update(selected)

# ── test / train_val 분리
test_mask        = full_df['patient_id'].isin(test_patient_ids)
test_df_all      = full_df[test_mask].reset_index(drop=True)
train_val_df_all = full_df[~test_mask].reset_index(drop=True)

def filter_by_channel(df, mode):
    if mode == 'r':
        return df[df['npy_file'].str.endswith('_rc.jpg.npy')].reset_index(drop=True)
    elif mode == 'g':
        return df[df['npy_file'].str.endswith('_gc.jpg.npy')].reset_index(drop=True)
    else:
        return df.reset_index(drop=True)

train_val_df = filter_by_channel(train_val_df_all, CHANNEL_MODE)
test_df      = filter_by_channel(test_df_all,      CHANNEL_MODE)

print(f"\n[{CHANNEL_MODE} 채널] Original full_df size: {len(full_df)}")
print(f"train_val_df size: {len(train_val_df)} ({len(train_val_df) / len(full_df):.2%})")
print(f"test_df size:      {len(test_df)} ({len(test_df) / len(full_df):.2%})")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold

random_state = 42
n_splits = 3

# ─────────────────────────────────────────────
# 환자 단위로 대표 레이블 추출 후 StratifiedKFold
# → 환자별로 클래스가 고정되어 있으므로 환자 단위 split으로도 클래스 보장 가능
# ─────────────────────────────────────────────

# 환자별 대표 레이블 (환자 하나는 하나의 클래스만 가짐)
patient_label_df = (
    train_val_df.groupby('patient_id')['temp_label']
    .first()
    .reset_index()
)  # columns: patient_id, temp_label

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

fold_indices = []  # (train_sample_idx, val_sample_idx) 리스트

for tr_p_idx, va_p_idx in skf.split(
    X=patient_label_df['patient_id'],
    y=patient_label_df['temp_label']
):
    tr_patients = set(patient_label_df.iloc[tr_p_idx]['patient_id'])
    va_patients = set(patient_label_df.iloc[va_p_idx]['patient_id'])

    tr_idx = train_val_df[train_val_df['patient_id'].isin(tr_patients)].index
    va_idx = train_val_df[train_val_df['patient_id'].isin(va_patients)].index

    fold_indices.append((tr_idx, va_idx))

# ─────────────────────────────────────────────
# 검증 출력
# ─────────────────────────────────────────────
print("=" * 60)
print("📋 Fold별 Patient Distribution 검증 (n_splits=5)")
print("=" * 60)

class_label_names = {0: 'Normal', 1: 'NAION', 2: 'ON'}
has_problem = False

for fold_i, (tr_idx, va_idx) in enumerate(fold_indices):
    tr_df = train_val_df.loc[tr_idx]
    va_df = train_val_df.loc[va_idx]

    tr_patients_by_class = tr_df.groupby('temp_label')['patient_id'].nunique()
    va_patients_by_class = va_df.groupby('temp_label')['patient_id'].nunique()

    print(f"\n[Fold {fold_i+1}]")
    print(f"  {'클래스':<10} {'Train 환자수':>12} {'Val 환자수':>12} {'Val 샘플수':>12}")
    print(f"  {'-'*48}")

    for cls_idx, cls_name in class_label_names.items():
        tr_p = tr_patients_by_class.get(cls_idx, 0)
        va_p = va_patients_by_class.get(cls_idx, 0)
        va_s = (va_df['temp_label'] == cls_idx).sum()
        warn = " ⚠️ 없음!" if va_p == 0 else ""
        print(f"  {cls_name:<10} {tr_p:>12} {va_p:>12} {va_s:>12}{warn}")
        if va_p == 0:
            has_problem = True

    overlap = len(set(tr_df['patient_id']) & set(va_df['patient_id']))
    if overlap > 0:
        print(f"  ❌ 환자 누수 발생: {overlap}명")
        has_problem = True
    else:
        print(f"  ✅ 환자 누수 없음")

print("\n" + "=" * 60)
if has_problem:
    print("❌ 문제 발견: 일부 Fold에 클래스 누락 또는 환자 누수가 있습니다.")
else:
    print("✅ 모든 Fold에 클래스가 고르게 분포되어 있습니다.")
print("=" * 60)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import StratifiedKFold
import copy
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

# ─────────────────────────────────────────────
# 기본 설정
# ─────────────────────────────────────────────
class_names = ['Normal', 'NAION', 'ON']
all_class_labels = list(range(num_classes))

class_weights = torch.tensor([1.0, 1.0, 5.0], device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

trained_models_weights = []
history = {
    'train_loss': [],
    'val_loss': [],
    'train_acc': [],
    'val_acc': [],
    'lr': []
}

# ─────────────────────────────────────────────
# fold_indices 생성 (환자 단위 StratifiedKFold)
# ─────────────────────────────────────────────
patient_label_df = (
    train_val_df.groupby('patient_id')['temp_label']
    .first()
    .reset_index()
)

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

fold_indices = []
for tr_p_idx, va_p_idx in skf.split(
    X=patient_label_df['patient_id'],
    y=patient_label_df['temp_label']
):
    tr_patients = set(patient_label_df.iloc[tr_p_idx]['patient_id'])
    va_patients = set(patient_label_df.iloc[va_p_idx]['patient_id'])
    tr_idx = train_val_df[train_val_df['patient_id'].isin(tr_patients)].index
    va_idx = train_val_df[train_val_df['patient_id'].isin(va_patients)].index
    fold_indices.append((tr_idx, va_idx))

print(f'\n--- 🏁 {n_splits}-Fold Cross Validation 학습 시작 ---')

num_workers = 0
loader_kwargs = {
    "batch_size": batch_size,
    "pin_memory": str(device).startswith("cuda"),
    "num_workers": num_workers
}

if num_workers > 0:
    loader_kwargs["persistent_workers"] = True
    loader_kwargs["prefetch_factor"] = 2

# ─────────────────────────────────────────────
# Fold Loop
# ─────────────────────────────────────────────
for fold, (train_idx, val_idx) in enumerate(fold_indices):
    print(f'\n========== Fold {fold + 1}/{n_splits} ==========')

    tr_df = train_val_df.loc[train_idx]
    va_df = train_val_df.loc[val_idx]

    # 환자 누수 확인
    train_patients = set(tr_df['patient_id'])
    val_patients   = set(va_df['patient_id'])
    patient_overlap = len(train_patients & val_patients)
    if patient_overlap != 0:
        print(f"⚠️ Warning: Fold {fold + 1} patient overlap = {patient_overlap}")

    # 클래스 누락 확인
    val_counts = va_df['temp_label'].value_counts().sort_index()
    missing_val_classes = sorted(set(all_class_labels) - set(val_counts.index))
    if missing_val_classes:
        print(f"⚠️ Warning: Fold {fold + 1} validation set에 없는 클래스: {missing_val_classes}")

    # ─────────────────────────────────────────────
    # DataLoader
    # ─────────────────────────────────────────────
    t_loader = DataLoader(
        FundusDataset(tr_df, x_dir, train_transform),
        shuffle=True,
        **loader_kwargs
    )

    v_loader = DataLoader(
        FundusDataset(va_df, x_dir, val_transform),
        shuffle=False,
        **loader_kwargs
    )

    # ─────────────────────────────────────────────
    # Model
    # ─────────────────────────────────────────────
    model = EfficientNetB0(num_classes=num_classes).to(device)
    model = transplant_weights(model)
    model = model.to(device)

    UNFREEZE_LAYERS = {'features.6', 'features.7', 'features.8', 'classifier'}

    for name, param in model.named_parameters():
        param.requires_grad = any(name.startswith(l) for l in UNFREEZE_LAYERS)

    total  = sum(p.numel() for p in model.parameters())
    frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"전체 파라미터: {total:,}")
    print(f"동결 파라미터: {frozen:,} ({frozen/total*100:.1f}%)")
    print(f"학습 파라미터: {total-frozen:,} ({(total-frozen)/total*100:.1f}%)")

    # ─────────────────────────────────────────────
    # 차등 학습률
    # ─────────────────────────────────────────────
    backbone_params   = [p for n, p in model.named_parameters()
                         if any(n.startswith(l) for l in {'features.6', 'features.7', 'features.8'})
                         and p.requires_grad]
    classifier_params = [p for n, p in model.named_parameters()
                         if n.startswith('classifier') and p.requires_grad]

    optimizer = torch.optim.Adam([
        {'params': backbone_params,   'lr': learning_rate * 0.1},
        {'params': classifier_params, 'lr': learning_rate},
    ])

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs,
        eta_min=1e-6
    )

    best_v_loss  = float('inf')
    best_weights = None

    fold_tl, fold_vl = [], []
    fold_ta, fold_va = [], []
    fold_lr = []

    # ─────────────────────────────────────────────
    # Epoch Loop
    # ─────────────────────────────────────────────
    for epoch in range(num_epochs):

        # ── Train ─────────────────────────────────
        model.train()
        train_loss_sum, train_correct, train_total = 0.0, 0, 0

        for inputs, targets in tqdm(
            t_loader,
            desc=f'Fold {fold+1}/{n_splits} | Epoch {epoch+1:02d}/{num_epochs} | Train',
            leave=False
        ):
            inputs  = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            outputs = model(inputs)
            loss    = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            batch_size_now  = targets.size(0)
            train_loss_sum += loss.item() * batch_size_now
            train_correct  += (outputs.argmax(dim=1) == targets).sum().item()
            train_total    += batch_size_now

        avg_train_loss = train_loss_sum / train_total
        avg_train_acc  = train_correct  / train_total

        # ── Validation ────────────────────────────
        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0

        with torch.inference_mode():
            for inputs, targets in tqdm(
                v_loader,
                desc=f'Fold {fold+1}/{n_splits} | Epoch {epoch+1:02d}/{num_epochs} | Val  ',
                leave=False
            ):
                inputs  = inputs.to(device, non_blocking=True)
                targets = targets.to(device, non_blocking=True)

                outputs = model(inputs)
                loss    = criterion(outputs, targets)
                preds   = outputs.argmax(dim=1)

                batch_size_now = targets.size(0)
                val_loss_sum  += loss.item() * batch_size_now
                val_correct   += (preds == targets).sum().item()
                val_total     += batch_size_now

        avg_val_loss = val_loss_sum / val_total
        avg_val_acc  = val_correct  / val_total
        cur_lr       = optimizer.param_groups[0]['lr']

        scheduler.step()

        fold_tl.append(avg_train_loss)
        fold_vl.append(avg_val_loss)
        fold_ta.append(avg_train_acc)
        fold_va.append(avg_val_acc)
        fold_lr.append(cur_lr)

        best_mark = ""
        if avg_val_loss < best_v_loss:
            best_v_loss  = avg_val_loss
            best_weights = copy.deepcopy(model.state_dict())
            best_mark    = " | best ✓"

        print(
            f"Fold {fold+1}/{n_splits} "
            f"Epoch {epoch+1:02d}/{num_epochs} | "
            f"Train Loss {avg_train_loss:.4f} Acc {avg_train_acc:.4f} | "
            f"Val Loss {avg_val_loss:.4f} Acc {avg_val_acc:.4f} | "
            f"LR {cur_lr:.6f}"
            f"{best_mark}"
        )

    # ─────────────────────────────────────────────
    # Fold 결과 저장
    # ─────────────────────────────────────────────
    history['train_loss'].append(fold_tl)
    history['val_loss'].append(fold_vl)
    history['train_acc'].append(fold_ta)
    history['val_acc'].append(fold_va)
    history['lr'].append(fold_lr)

    if best_weights is not None:
        trained_models_weights.append(best_weights)

        weight_save_dir = r"C:\Users\konyang\Desktop\2026_LAB\data6\model1"
        os.makedirs(weight_save_dir, exist_ok=True)

        weight_path = os.path.join(weight_save_dir, f"fold{fold+1}_best.pth")
        torch.save(best_weights, weight_path)
        print(f"💾 Fold {fold+1} weight 저장 완료: {weight_path}")

    print(
        f"✅ Fold {fold+1}/{n_splits} 완료 | "
        f"Best Val Loss: {best_v_loss:.4f}"
    )

print('\n✅ 모든 Fold 학습 완료!')
print(f"저장된 fold weight 수: {len(trained_models_weights)}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n_splits = len(history['train_loss'])
epochs = range(1, len(history['train_loss'][0]) + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('5-Fold Cross Validation 학습 결과', fontsize=15, fontweight='bold')

colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

# ── 1. Train Loss ────────────────────────────────
ax = axes[0, 0]
for fold in range(n_splits):
    ax.plot(epochs, history['train_loss'][fold], color=colors[fold], label=f'Fold {fold+1}')
ax.plot(epochs, np.mean(history['train_loss'], axis=0), 'k--', linewidth=2, label='Mean')
ax.set_title('Train Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── 2. Val Loss ──────────────────────────────────
ax = axes[0, 1]
for fold in range(n_splits):
    ax.plot(epochs, history['val_loss'][fold], color=colors[fold], label=f'Fold {fold+1}')
ax.plot(epochs, np.mean(history['val_loss'], axis=0), 'k--', linewidth=2, label='Mean')
ax.set_title('Validation Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── 3. Train Accuracy ────────────────────────────
ax = axes[1, 0]
for fold in range(n_splits):
    ax.plot(epochs, history['train_acc'][fold], color=colors[fold], label=f'Fold {fold+1}')
ax.plot(epochs, np.mean(history['train_acc'], axis=0), 'k--', linewidth=2, label='Mean')
ax.set_title('Train Accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── 4. Val Accuracy ──────────────────────────────
ax = axes[1, 1]
for fold in range(n_splits):
    ax.plot(epochs, history['val_acc'][fold], color=colors[fold], label=f'Fold {fold+1}')
ax.plot(epochs, np.mean(history['val_acc'], axis=0), 'k--', linewidth=2, label='Mean')
ax.set_title('Validation Accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(weight_save_dir, 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

# ── Fold별 Best Val Loss 요약 ────────────────────
print("\n=== Fold별 Best Val Loss 요약 ===")
best_val_losses = [min(history['val_loss'][f]) for f in range(n_splits)]
best_val_accs   = [max(history['val_acc'][f])  for f in range(n_splits)]
for fold in range(n_splits):
    print(f"  Fold {fold+1} | Best Val Loss: {best_val_losses[fold]:.4f} | Best Val Acc: {best_val_accs[fold]:.4f}")
print(f"  {'─'*50}")
print(f"  Mean   | Best Val Loss: {np.mean(best_val_losses):.4f} | Best Val Acc: {np.mean(best_val_accs):.4f}")

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# ─────────────────────────────────────────────
# 설정
# ─────────────────────────────────────────────
class_names = ['Normal', 'NAION', 'ON']
num_workers = 0

loader_kwargs = {
    "batch_size": batch_size,
    "pin_memory": str(device).startswith("cuda"),
    "num_workers": num_workers
}

# ─────────────────────────────────────────────
# Test DataLoader
# ─────────────────────────────────────────────
test_loader = DataLoader(
    FundusDataset(test_df, x_dir, val_transform),
    shuffle=False,
    **loader_kwargs
)

# ─────────────────────────────────────────────
# 5-Fold 앙상블 inference
# ─────────────────────────────────────────────
print(f"🔍 앙상블 Inference 시작 (총 {len(trained_models_weights)}개 모델)")

all_fold_probs = []  # 각 fold의 softmax 확률 저장
all_targets    = []  # 정답 레이블 (첫 번째 fold에서만 수집)

for fold_idx, weights in enumerate(trained_models_weights):
    model = EfficientNetB0(num_classes=num_classes).to(device)
    model.load_state_dict(weights)
    model.eval()

    fold_probs = []
    fold_targets = []

    with torch.inference_mode():
        for inputs, targets in test_loader:
            inputs  = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            outputs = model(inputs)
            probs   = torch.softmax(outputs, dim=1)  # (B, num_classes)

            fold_probs.append(probs.cpu())
            if fold_idx == 0:
                fold_targets.append(targets.cpu())

    all_fold_probs.append(torch.cat(fold_probs, dim=0))   # (N, num_classes)
    if fold_idx == 0:
        all_targets = torch.cat(fold_targets, dim=0)       # (N,)

    print(f"  ✅ Fold {fold_idx + 1} inference 완료")

# ─────────────────────────────────────────────
# 앙상블: fold별 확률 평균 → 최종 예측
# ─────────────────────────────────────────────
stacked_probs = torch.stack(all_fold_probs, dim=0)  # (n_folds, N, num_classes)
final_probs   = stacked_probs.mean(dim=0)            # (N, num_classes)
final_preds   = final_probs.argmax(dim=1)            # (N,)

y_true = all_targets.numpy()
y_pred = final_preds.numpy()

# ─────────────────────────────────────────────
# 성능 지표 출력
# ─────────────────────────────────────────────
acc = (y_true == y_pred).mean()
print(f"\n📊 Test Accuracy (앙상블): {acc:.4f} ({acc*100:.2f}%)")
print("\n=== Classification Report ===")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# ─────────────────────────────────────────────
# Confusion Matrix 시각화
# ─────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # 행 단위 정규화

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Ensemble Test Result  (Acc: {acc:.4f})', fontsize=13, fontweight='bold')

# 절댓값 CM
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title('Confusion Matrix (Count)')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

# 정규화 CM
sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title('Confusion Matrix (Normalized)')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig(os.path.join(weight_save_dir, 'ensemble_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from torch.utils.data import DataLoader
from PIL import Image

# ─────────────────────────────────────────────
# Grad-CAM 클래스
# ─────────────────────────────────────────────
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, input_tensor, target_class):
        self.model.zero_grad()
        output = self.model(input_tensor)                      # (1, num_classes)
        score = output[0, target_class]
        score.backward()

        # 가중치 평균
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # (1, 1, H, W)
        cam = torch.relu(cam).squeeze().cpu().numpy()

        # 정규화
        cam -= cam.min()
        if cam.max() > 0:
            cam /= cam.max()
        return cam, output.detach()


def overlay_cam(img_np, cam, alpha=0.5):
    """원본 이미지에 CAM 히트맵 오버레이"""
    h, w = img_np.shape[:2]
    cam_resized = np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize((w, h), Image.BILINEAR)) / 255.0
    heatmap = cm.jet(cam_resized)[:, :, :3]  # (H, W, 3), RGB
    overlay = (1 - alpha) * img_np / 255.0 + alpha * heatmap
    overlay = np.clip(overlay, 0, 1)
    return overlay, cam_resized


# ─────────────────────────────────────────────
# 설정
# ─────────────────────────────────────────────
class_names  = ['Normal', 'NAION', 'ON']
SAMPLES_PER_CLASS = 3   # 클래스당 시각화 샘플 수
ALPHA = 0.5             # CAM 투명도

# ─────────────────────────────────────────────
# 클래스별 샘플 추출 (test_df 기준)
# ─────────────────────────────────────────────
sample_indices = []
for cls_idx in range(len(class_names)):
    cls_mask = test_df['temp_label'] == cls_idx
    cls_indices = test_df[cls_mask].index.tolist()
    selected = cls_indices[:SAMPLES_PER_CLASS]
    sample_indices.extend([(idx, cls_idx) for idx in selected])

sample_dataset = FundusDataset(test_df, x_dir, val_transform)

# ─────────────────────────────────────────────
# 앙상블 Grad-CAM: fold별 CAM 평균
# ─────────────────────────────────────────────
# 결과 저장: {sample_idx: {'cams': [], 'pred_probs': [], 'true_label': int}}
results = {idx: {'cams': [], 'pred_probs': [], 'true_label': true_cls}
           for idx, true_cls in sample_indices}

for fold_idx, weights in enumerate(trained_models_weights):
    # 모델 로드
    model = EfficientNetB0(num_classes=num_classes).to(device)
    model.load_state_dict(weights)
    model.eval()

    # EfficientNetB0의 마지막 conv layer 지정
    target_layer = model.features[-1]
    grad_cam = GradCAM(model, target_layer)

    for img_idx, true_cls in sample_indices:
        input_tensor, _ = sample_dataset[img_idx]
        input_tensor = input_tensor.unsqueeze(0).to(device)
        input_tensor.requires_grad_(True)

        cam, output = grad_cam.generate(input_tensor, target_class=true_cls)
        probs = torch.softmax(output, dim=1).cpu().numpy()[0]

        results[img_idx]['cams'].append(cam)
        results[img_idx]['pred_probs'].append(probs)

    print(f"  ✅ Fold {fold_idx + 1} Grad-CAM 완료")

# ─────────────────────────────────────────────
# 시각화
# ─────────────────────────────────────────────
n_samples = len(sample_indices)
fig, axes = plt.subplots(n_samples, 3, figsize=(15, n_samples * 4))
fig.suptitle('Grad-CAM Visualization (Ensemble Average)', fontsize=14, fontweight='bold')

# val_transform의 역정규화용
MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

for row, (img_idx, true_cls) in enumerate(sample_indices):
    # 앙상블 평균 CAM & 예측 확률
    avg_cam   = np.mean(results[img_idx]['cams'], axis=0)
    avg_probs = np.mean(results[img_idx]['pred_probs'], axis=0)
    pred_cls  = int(np.argmax(avg_probs))

    # 원본 이미지 복원 (역정규화)
    img_tensor, _ = sample_dataset[img_idx]
    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = (img_np * STD + MEAN)
    img_np = np.clip(img_np * 255, 0, 255).astype(np.uint8)

    overlay, cam_resized = overlay_cam(img_np, avg_cam, alpha=ALPHA)

    correct = "✅" if pred_cls == true_cls else "❌"
    title_base = (f"True: {class_names[true_cls]} | "
                  f"Pred: {class_names[pred_cls]} {correct}\n"
                  f"{' / '.join([f'{class_names[i]}:{avg_probs[i]:.2f}' for i in range(len(class_names))])}")

    # 원본
    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title(f"Original\n{title_base}", fontsize=8)
    axes[row, 0].axis('off')

    # CAM 히트맵
    axes[row, 1].imshow(cam_resized, cmap='jet')
    axes[row, 1].set_title("Grad-CAM Heatmap", fontsize=8)
    axes[row, 1].axis('off')

    # 오버레이
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title("Overlay", fontsize=8)
    axes[row, 2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(weight_save_dir, 'gradcam_ensemble.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"\n💾 저장 완료: {os.path.join(weight_save_dir, 'gradcam_ensemble.png')}")

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm

# =========================
# 경로 설정
# =========================
INPUT_DIR  = Path(r"C:\Users\konyang\Desktop\2026_LAB\data6/dataset/X")   # .npy 파일이 있는 폴더
OUTPUT_DIR = Path(r"C:\Users\konyang\Desktop\2026_LAB\data6/dataset/X_imporve")  # 결과 저장 폴더

SHRINK_RATIO = 0.97
THRESH       = 10


def remove_protruded_border(img, shrink_ratio=0.97, thresh=10):
    """단안 이미지에 원형 마스크 적용 (uint8 BGR/RGB 모두 호환)"""
    if img is None:
        return None

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)  # npy는 RGB 저장이 일반적

    _, binary = cv2.threshold(gray, thresh, 255, cv2.THRESH_BINARY)

    kernel = np.ones((7, 7), np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  kernel)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return img

    largest = max(contours, key=cv2.contourArea)
    (x, y), radius = cv2.minEnclosingCircle(largest)
    center = (int(x), int(y))
    radius = int(radius * shrink_ratio)

    if radius <= 0:
        return img

    mask = np.zeros(gray.shape, dtype=np.uint8)
    cv2.circle(mask, center, radius, 255, -1)

    result = cv2.bitwise_and(img, img, mask=mask)
    return result


def remove_protruded_border_merged(img, shrink_ratio=0.97, thresh=10):
    """좌안+우안 합쳐진 (H, W*2, C) 배열에 각각 원형 마스크 적용"""
    h, w = img.shape[:2]
    mid = w // 2

    left_eye  = img[:, :mid, :]
    right_eye = img[:, mid:, :]

    left_processed  = remove_protruded_border(left_eye,  shrink_ratio, thresh)
    right_processed = remove_protruded_border(right_eye, shrink_ratio, thresh)

    # 마스크 실패 시 원본 유지
    if left_processed  is None: left_processed  = left_eye
    if right_processed is None: right_processed = right_eye

    return np.concatenate([left_processed, right_processed], axis=1)


def process_npy_files(input_dir, output_dir, shrink_ratio=0.97, thresh=10):
    input_dir  = Path(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    npy_paths = sorted(input_dir.glob("*.npy"))

    if not npy_paths:
        print("처리할 .npy 파일이 없습니다.")
        return

    print(f"총 {len(npy_paths)}개 파일 처리 시작")

    failed = []
    for npy_path in tqdm(npy_paths, desc="Circular Mask 적용"):
        try:
            img = np.load(str(npy_path))  # (H, W*2, C), uint8

            # shape 검증
            if img.ndim != 3:
                raise ValueError(f"예상치 못한 shape: {img.shape}")

            processed = remove_protruded_border_merged(img, shrink_ratio, thresh)

            save_path = output_dir / npy_path.name
            np.save(str(save_path), processed)  # dtype 유지 (uint8)

        except Exception as e:
            print(f"\n[오류] {npy_path.name}: {e}")
            failed.append(npy_path.name)

    print(f"\n완료. 성공: {len(npy_paths) - len(failed)}개 / 실패: {len(failed)}개")
    if failed:
        print("실패 파일:", failed)


if __name__ == "__main__":
    process_npy_files(
        input_dir    = INPUT_DIR,
        output_dir   = OUTPUT_DIR,
        shrink_ratio = SHRINK_RATIO,
        thresh       = THRESH
    )

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# ─────────────────────────────────────────────
# 이 셀 바로 위의 앙상블 inference 셀에서
# y_true, y_pred 가 이미 정의되어 있어야 합니다.
# ─────────────────────────────────────────────

class_names = ['Normal', 'NAION', 'ON']
num_classes  = len(class_names)

# ─────────────────────────────────────────────
# Confusion Matrix 기반 지표 계산
# ─────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)   # shape: (3, 3)

metrics = {}
for i, cls in enumerate(class_names):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP          # 다른 클래스를 i로 잘못 예측
    FN = cm[i, :].sum() - TP          # i 클래스를 다른 클래스로 잘못 예측
    TN = cm.sum() - TP - FP - FN

    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0   # Recall
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    ppv         = TP / (TP + FP) if (TP + FP) > 0 else 0.0   # Precision
    npv         = TN / (TN + FN) if (TN + FN) > 0 else 0.0
    accuracy    = (TP + TN) / cm.sum()

    metrics[cls] = {
        'Accuracy':    accuracy,
        'PPV':         ppv,
        'NPV':         npv,
        'Sensitivity': sensitivity,
        'Specificity': specificity,
    }

# 전체(macro average) 추가
overall_acc = accuracy_score(y_true, y_pred)
for metric in ['Accuracy', 'PPV', 'NPV', 'Sensitivity', 'Specificity']:
    metrics['Macro Avg'] = metrics.get('Macro Avg', {})
    metrics['Macro Avg'][metric] = np.mean([metrics[c][metric] for c in class_names])

# ─────────────────────────────────────────────
# 콘솔 출력
# ─────────────────────────────────────────────
metric_keys = ['Accuracy', 'PPV', 'NPV', 'Sensitivity', 'Specificity']
metric_desc = {
    'Accuracy':    '전체 정답률',
    'PPV':         '결절 예측 정확도',
    'NPV':         '비결절 예측 정확도',
    'Sensitivity': '결절 탐지율 (Recall)',
    'Specificity': '비결절 탐지율',
}

header = f"{'Metric':<14} {'설명':<22} " + "  ".join(f"{c:>12}" for c in class_names) + f"  {'Macro Avg':>12}"
print("=" * len(header))
print(header)
print("=" * len(header))

for mk in metric_keys:
    row = f"{mk:<14} {metric_desc[mk]:<22} "
    for cls in class_names:
        row += f"  {metrics[cls][mk]:>12.4f}"
    row += f"  {metrics['Macro Avg'][mk]:>12.4f}"
    print(row)

print("=" * len(header))
print(f"\n전체 Accuracy: {overall_acc:.4f}")
print("\n[Classification Report]")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# ─────────────────────────────────────────────
# 슬라이드 스타일 테이블 시각화
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))
ax.axis('off')

col_labels = ['Metric', '설명'] + class_names + ['Macro Avg']
rows = []
for mk in metric_keys:
    row = [mk, metric_desc[mk]]
    row += [f"{metrics[cls][mk]:.3f}" for cls in class_names]
    row += [f"{metrics['Macro Avg'][mk]:.3f}"]
    rows.append(row)

table = ax.table(
    cellText=rows,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2.0)

# 헤더 스타일
for j in range(len(col_labels)):
    table[0, j].set_facecolor('#1e3a5f')
    table[0, j].set_text_props(color='white', fontweight='bold')

# 짝수 행 배경
for i in range(1, len(rows) + 1):
    for j in range(len(col_labels)):
        if i % 2 == 0:
            table[i, j].set_facecolor('#f0f4f8')
        else:
            table[i, j].set_facecolor('#ffffff')

# 클래스 열 색상 강조
cls_colors = {'Normal': '#d4edda', 'NAION': '#f8d7da', 'ON': '#fff3cd'}
for ci, cls in enumerate(class_names):
    col_idx = 2 + ci
    for ri in range(1, len(rows) + 1):
        table[ri, col_idx].set_facecolor(cls_colors[cls])

ax.set_title('모델 성능지표 (EfficientNet-B0 앙상블)', fontsize=14,
             fontweight='bold', pad=20, color='#1e3a5f')

plt.tight_layout()
plt.savefig('performance_metrics_table.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()
print("✅ 저장 완료: performance_metrics_table.png")


In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform

# ─────────────────────────────────────────────
# 한글 폰트 설정 (그래프 내 한글 깨짐 방지)
# ─────────────────────────────────────────────
def set_korean_font():
    system_name = platform.system()
    if system_name == 'Windows':
        font_name = 'Malgun Gothic'
    elif system_name == 'Darwin':  # macOS
        font_name = 'AppleGothic'
    else:  # Linux
        installed = {f.name for f in fm.fontManager.ttflist}
        # 우선순위: 나눔고딕 > Noto Sans CJK KR > Noto Sans CJK JP(한글 글리프 포함)
        candidates = ['NanumGothic', 'Noto Sans CJK KR', 'Noto Sans CJK JP', 'Noto Sans KR']
        font_name = next((f for f in candidates if f in installed), None)

    if font_name:
        plt.rcParams['font.family'] = font_name
    plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

set_korean_font()

# ─────────────────────────────────────────────
# 이 셀 바로 위의 앙상블 inference 셀에서
# y_true, y_pred 가 이미 정의되어 있어야 합니다.
# ─────────────────────────────────────────────

class_names = ['Normal', 'NAION', 'ON']
num_classes  = len(class_names)

# ─────────────────────────────────────────────
# Confusion Matrix 기반 지표 계산
# ─────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)   # shape: (3, 3)

metrics = {}
for i, cls in enumerate(class_names):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP          # 다른 클래스를 i로 잘못 예측
    FN = cm[i, :].sum() - TP          # i 클래스를 다른 클래스로 잘못 예측
    TN = cm.sum() - TP - FP - FN

    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0   # Recall
    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0
    ppv         = TP / (TP + FP) if (TP + FP) > 0 else 0.0   # Precision
    npv         = TN / (TN + FN) if (TN + FN) > 0 else 0.0
    accuracy    = (TP + TN) / cm.sum()

    metrics[cls] = {
        'Accuracy':    accuracy,
        'PPV':         ppv,
        'NPV':         npv,
        'Sensitivity': sensitivity,
        'Specificity': specificity,
    }

# 전체(macro average) 추가
overall_acc = accuracy_score(y_true, y_pred)
for metric in ['Accuracy', 'PPV', 'NPV', 'Sensitivity', 'Specificity']:
    metrics['Macro Avg'] = metrics.get('Macro Avg', {})
    metrics['Macro Avg'][metric] = np.mean([metrics[c][metric] for c in class_names])

# ─────────────────────────────────────────────
# 콘솔 출력
# ─────────────────────────────────────────────
metric_keys = ['Accuracy', 'PPV', 'NPV', 'Sensitivity', 'Specificity']
metric_desc = {
    'Accuracy':    '전체 정답률',
    'PPV':         '결절 예측 정확도',
    'NPV':         '비결절 예측 정확도',
    'Sensitivity': '결절 탐지율 (Recall)',
    'Specificity': '비결절 탐지율',
}

header = f"{'Metric':<14} {'설명':<22} " + "  ".join(f"{c:>12}" for c in class_names) + f"  {'Macro Avg':>12}"
print("=" * len(header))
print(header)
print("=" * len(header))

for mk in metric_keys:
    row = f"{mk:<14} {metric_desc[mk]:<22} "
    for cls in class_names:
        row += f"  {metrics[cls][mk]:>12.4f}"
    row += f"  {metrics['Macro Avg'][mk]:>12.4f}"
    print(row)

print("=" * len(header))
print(f"\n전체 Accuracy: {overall_acc:.4f}")
print("\n[Classification Report]")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# ─────────────────────────────────────────────
# 슬라이드 스타일 테이블 시각화
# ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 4))
ax.axis('off')

col_labels = ['Metric', '설명'] + class_names + ['Macro Avg']
rows = []
for mk in metric_keys:
    row = [mk, metric_desc[mk]]
    row += [f"{metrics[cls][mk]:.3f}" for cls in class_names]
    row += [f"{metrics['Macro Avg'][mk]:.3f}"]
    rows.append(row)

table = ax.table(
    cellText=rows,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2.0)

# 헤더 스타일
for j in range(len(col_labels)):
    table[0, j].set_facecolor('#1e3a5f')
    table[0, j].set_text_props(color='white', fontweight='bold')

# 짝수 행 배경
for i in range(1, len(rows) + 1):
    for j in range(len(col_labels)):
        if i % 2 == 0:
            table[i, j].set_facecolor('#f0f4f8')
        else:
            table[i, j].set_facecolor('#ffffff')

# 클래스 열 색상 강조
cls_colors = {'Normal': '#d4edda', 'NAION': '#f8d7da', 'ON': '#fff3cd'}
for ci, cls in enumerate(class_names):
    col_idx = 2 + ci
    for ri in range(1, len(rows) + 1):
        table[ri, col_idx].set_facecolor(cls_colors[cls])

ax.set_title('모델 성능지표 (EfficientNet-B0 앙상블)', fontsize=14,
             fontweight='bold', pad=20, color='#1e3a5f')

plt.tight_layout()
plt.savefig('performance_metrics_table.png', dpi=150, bbox_inches='tight',
            facecolor='white')
plt.show()
print("✅ 저장 완료: performance_metrics_table.png")
